# 🤖 Procesamiento de Lenguaje Natural

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Módulo 6 — Chatbot RAG con Anti-Alucinación

---

> **Actividad 6:** Implementación de un chatbot con sistema RAG (Recuperación Aumentada por Generación) para responder preguntas técnicas sobre hojas de referencia de Python y Machine Learning, con énfasis en el manejo de contenido tabular y prevención de alucinaciones.


## 👥 Equipo

* **Nombres y matrículas:**

  *   Jose Angel Barajas — A01797221
  *   Elemento de lista
  *   Elemento de lista

* **Número de Equipo:**


---

## 📋 v6 — RAG Anti-Alucinación con Grok API + Extracción de Tablas

### Historial de versiones

| Versión | LLM | Cambio principal |
|---------|-----|-----------------|
| v1–v3 | `qwen2.5-coder-7b` (local) | Pruebas iniciales, prompts fallidos con modelos pequeños |
| v4 | `qwen2.5-coder-7b` (local) | RetrievalQA sin prompt custom — funcionaba pero sin control |
| v5 | `grok-4-1-fast-non-reasoning` | API xAI, threshold=0.0, prompt suavizado, ChromaDB |
| **v6** | **`grok-3-mini-fast`** | **pdfplumber para tablas, MMR retrieval, grounding check, preguntas requeridas** |

### Cobertura de criterios de evaluación

| Criterio | Cómo se cubre en v6 |
|----------|---------------------|
| **Implementación del chatbot** | Chatbot funcional con Gradio + modo notebook |
| **Integración de RAG** | ChromaDB + MMR retrieval + historial conversacional |
| **Manejo de documentos PDF** | PyPDFLoader (texto) + pdfplumber (tablas en Markdown) |
| **Base vectorial** | ChromaDB con `all-MiniLM-L6-v2`, threshold dinámico |
| **Prompts / LLM** | Prompt calibrado + temperatura 0.1 para máxima fidelidad |
| **Calidad de respuestas** | Grounding Check automático post-respuesta |
| **Preguntas requeridas** | 6 preguntas (a–f) ejecutadas y documentadas |
| **Manejo de alucinaciones** | MMR + grounding check + prompt estricto + umbral de confianza |
| **Análisis técnico** | Visualización de scores, chunks y métricas del retrieval |
| **Conclusiones** | Análisis de retos tabulares + conclusiones finales |
| **Documentación** | Markdowns detallados en cada sección |


---

## 🧩 Step 1 — Instalación de Paquetes

**Paquetes clave:**
- `pdfplumber` → Extracción de tablas en formato Markdown/texto estructurado *(nuevo en v6)*
- `langchain-community` → Loaders, VectorStores, retrievers
- `chromadb` → Base de datos vectorial persistente
- `sentence-transformers` → Embeddings locales (all-MiniLM-L6-v2)
- `gradio` → Interfaz de chat web interactiva


In [1]:
import sys

# Core RAG stack
!{sys.executable} -m pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-openai langchain-core -q
!{sys.executable} -m pip install chromadb -q
!{sys.executable} -m pip install pypdf -q
!{sys.executable} -m pip install sentence-transformers -q
!{sys.executable} -m pip install gradio -q
!{sys.executable} -m pip install openai requests python-dotenv -q
!{sys.executable} -m pip install pdfplumber -q
!{sys.executable} -m pip install huggingface_hub -q

print("✅ Todos los paquetes instalados.")


✅ Todos los paquetes instalados.


---

## ⚙️ Step 2 — Imports y Configuración

Se importan las librerías necesarias y se carga la clave de API desde el archivo `.env`.

> **Nota sobre seguridad:** La API key se carga desde una variable de entorno (`xAI_API_KEY`) para evitar exponer credenciales en el código.


In [2]:
# ─── Document loading & PDF parsing ────────────────────────────────────────
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# ─── Vector store & embeddings ──────────────────────────────────────────────
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# ─── Prompt engineering ──────────────────────────────────────────────────────
from langchain_core.prompts import PromptTemplate

# ─── Standard libraries ──────────────────────────────────────────────────────
import requests
import os
import re
import json as _json
import warnings
import pdfplumber
import numpy as np
from collections import Counter
from dotenv import load_dotenv
import pathlib

# ─── UI ──────────────────────────────────────────────────────────────────────
import gradio as gr

# ─── Suppress minor warnings ─────────────────────────────────────────────────
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ─────────────────────────────────────────────────────────────────────────────
# Carga robusta de variables de entorno
# ─────────────────────────────────────────────────────────────────────────────
notebook_dir = pathlib.Path().resolve()
for _candidate in [notebook_dir / ".env", notebook_dir / "env"]:
    if _candidate.exists():
        load_dotenv(dotenv_path=str(_candidate), override=True)
        print(f"✅ Variables cargadas desde: {_candidate.name}")
        break

# ─── Leer keys disponibles ───────────────────────────────────────────────────
HF_TOKEN   = os.getenv("HF_TOKEN")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
GROK_KEY   = os.getenv("xAI_API_KEY")

if GEMINI_KEY and not GEMINI_KEY.startswith("AIza"):
    print("⚠️  GEMINI_API_KEY no tiene formato válido (debe empezar con 'AIza') — ignorada.")
    GEMINI_KEY = None

# ─────────────────────────────────────────────────────────────────────────────
# Configuración de Ollama (proveedor local, sin API key)
# ─────────────────────────────────────────────────────────────────────────────
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = "llama3.2"   # Cambiar si se usa otro modelo (ej: "mistral", "gemma2")

def _ollama_available():
    """Verifica si Ollama está corriendo y tiene el modelo disponible."""
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3)
        if r.status_code != 200:
            return False
        models = [m["name"].split(":")[0] for m in r.json().get("models", [])]
        return OLLAMA_MODEL.split(":")[0] in models
    except Exception:
        return False

OLLAMA_OK = _ollama_available()

# ─── Configuración de todos los proveedores ───────────────────────────────────
HF_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

_providers = {
    "ollama": {
        "key":   "local",          # No necesita key real
        "model": OLLAMA_MODEL,
        "url":   f"{OLLAMA_BASE_URL}/v1/chat/completions",  # OpenAI-compatible
        "label": f"Ollama LOCAL [{OLLAMA_MODEL}]",
        "available": OLLAMA_OK,
    },
    "huggingface": {
        "key":   HF_TOKEN,
        "model": HF_MODEL,
        "url":   f"https://router.huggingface.co/hf-inference/models/{HF_MODEL}/v1/chat/completions",
        "label": "HuggingFace Inference API",
        "available": bool(HF_TOKEN),
    },
    "gemini": {
        "key":   GEMINI_KEY,
        "model": "gemini-3.5-flash",
        "url":   "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions",
        "label": "Google Gemini 2.0 Flash",
        "available": bool(GEMINI_KEY),
    },
    "grok": {
        "key":   GROK_KEY,
        "model": "grok-3-mini-fast",
        "url":   "https://api.x.ai/v1/chat/completions",
        "label": "xAI Grok 3 Mini Fast",
        "available": bool(GROK_KEY),
    },
}

# ─── Selección automática en orden de prioridad ───────────────────────────────
PROVIDER_ORDER = ["ollama", "huggingface", "gemini", "grok"]

LLM_PROVIDER = LLM_MODEL = LLM_URL = LLM_KEY = None
for _name in PROVIDER_ORDER:
    _cfg = _providers[_name]
    if _cfg["available"]:
        LLM_PROVIDER = _name
        LLM_MODEL    = _cfg["model"]
        LLM_URL      = _cfg["url"]
        LLM_KEY      = _cfg["key"]
        break

# ─── Reporte de estado ────────────────────────────────────────────────────────
print()
print("─" * 60)
print("  Estado de proveedores LLM:")
for _name in PROVIDER_ORDER:
    _cfg = _providers[_name]
    status = "✅ Disponible" if _cfg["available"] else "❌ No disponible"
    active = " ← ACTIVO" if _name == LLM_PROVIDER else ""
    print(f"  {status}  {_cfg['label']}{active}")
print("─" * 60)

if not LLM_PROVIDER:
    print()
    print("  ❌ NINGÚN PROVEEDOR DISPONIBLE")
    print()
    print("  Solución más rápida — Ollama local (sin API key):")
    print("    1. brew install ollama")
    print("    2. ollama serve        (en otra terminal)")
    print("    3. ollama pull llama3.2")
    print("    4. Re-ejecuta esta celda")
    print()
    print("  Alternativa — HuggingFace (gratuito):")
    print("    1. Crea cuenta en huggingface.co/join")
    print("    2. Genera token en huggingface.co/settings/tokens")
    print("    3. Agrega HF_TOKEN=hf_... al archivo .env")


/var/folders/n5/sy8lrn6n66j27hcdc7r_l_kc0000gn/T/ipykernel_11351/4256176648.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/rolandooviedo/Documents/Master/nlp_torch/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Variables cargadas desde: .env

────────────────────────────────────────────────────────────
  Estado de proveedores LLM:
  ❌ No disponible  Ollama LOCAL [llama3.2]
  ❌ No disponible  HuggingFace Inference API
  ✅ Disponible  Google Gemini 2.0 Flash ← ACTIVO
  ✅ Disponible  xAI Grok 3 Mini Fast
────────────────────────────────────────────────────────────


---

## 🧠 Step 3 — Configuración del LLM

El sistema soporta **4 proveedores** en orden de prioridad. Usa el primero disponible:

| Prioridad | Proveedor | Costo | Req. |
|-----------|-----------|-------|------|
| 1️⃣ | **Ollama (local)** | 🆓 Gratis | Instalar Ollama + descargar modelo |
| 2️⃣ | **HuggingFace** | 🆓 Gratis | Cuenta HF + token |
| 3️⃣ | **Google Gemini** | 🆓 Gratis | Clave AIza válida con cuota habilitada |
| 4️⃣ | **xAI Grok** | 💳 Pago | Créditos en console.x.ai |

### ⭐ Opción recomendada: Ollama (local, sin internet, sin cuotas)

```bash
# 1. Instalar Ollama (solo una vez)
brew install ollama          # macOS con Homebrew

# 2. Iniciar el servidor Ollama (en una terminal separada)
ollama serve

# 3. Descargar un modelo (solo una vez, ~4 GB)
ollama pull llama3.2         # Recomendado: rápido y capaz en M3 Pro
```

No necesitas ninguna API key. El modelo corre 100% en tu M3 Pro con Metal.

### Alternativa: HuggingFace (sin instalación local)

1. Crea cuenta gratuita en [huggingface.co](https://huggingface.co/join)  
2. Ve a [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → **New token (Read)**
3. Agrega al archivo `.env`: `HF_TOKEN=hf_...`

### Sobre Gemini (Error 429 / quota=0)

El error `limit: 0` significa que el **proyecto de Google Cloud** asociado a tu key tiene billing deshabilitado.  
Para activar la cuota gratuita: ve a [console.cloud.google.com](https://console.cloud.google.com) → tu proyecto → **APIs & Services** → **Gemini API** → verifica que esté habilitada y que el billing esté activo.


In [3]:
def llm_chat(messages, temperature=0.1, max_tokens=2048):
    """
    Llama al LLM activo usando protocolo OpenAI Chat Completions.
    Soporta: Ollama (local), HuggingFace, Gemini, Grok — con fallback automático.
    
    Args:
        messages: Lista de dicts {'role': '...', 'content': '...'}
        temperature: 0.1 = muy fiel al contexto (recomendado para RAG)
        max_tokens: Máximo de tokens en la respuesta
    Returns:
        str: Respuesta del modelo
    """
    if not LLM_PROVIDER:
        raise ValueError(
            "Ningún proveedor LLM disponible.\n"
            "Opción más rápida — Ollama local (sin API key):\n"
            "  1. brew install ollama\n"
            "  2. ollama serve         (en otra terminal)\n"
            "  3. ollama pull llama3.2\n"
            "  4. Re-ejecuta la celda de imports"
        )
    
    def _call_provider(name):
        cfg = _providers[name]
        if not cfg["available"]:
            raise ValueError(f"Proveedor {name} no disponible")
        
        headers = {"Content-Type": "application/json"}
        # Ollama local no necesita Authorization
        if name != "ollama":
            headers["Authorization"] = f"Bearer {cfg['key']}"
        
        payload = {
            "model":       cfg["model"],
            "messages":    messages,
            "temperature": temperature,
            "max_tokens":  max_tokens,
        }
        r = requests.post(cfg["url"], headers=headers, json=payload, timeout=120)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]
    
    #Intentar en orden con fallback:
    tried = []
    for name in PROVIDER_ORDER:
        if not _providers[name]["available"]:
            continue
        try:
            result = _call_provider(name)
            if name != LLM_PROVIDER:
                print(f"   ℹ️  Fallback activo: {_providers[name]['label']}")
            return result
        except requests.exceptions.HTTPError as e:
            code = e.response.status_code if e.response is not None else 0
            msg  = e.response.text[:100] if e.response is not None else str(e)
            tried.append(f"  {name} [{code}]: {msg}")
            if code in (429, 403, 401):
                continue  # Cuota/auth → probar siguiente
            raise
        except Exception as e:
            tried.append(f"  {name}: {str(e)[:80]}")
            continue
    
    raise RuntimeError("Todos los proveedores fallaron:\n" + "\n".join(tried))

# Alias para compatibilidad
grok_chat = llm_chat


# ─── Test de conexión ─────────────────────────────────────────────────────────
if LLM_PROVIDER:
    print(f"\nProbando conexión con {_providers[LLM_PROVIDER]['label']}...")
    try:
        resp = llm_chat([{"role":"user","content":"Reply with exactly: Connection OK"}])
        print(f"✅ {resp[:100]}")
    except Exception as e:
        print(f"❌ Error: {e}")
else:
    print("\n⚠️  Configura un proveedor LLM para continuar (ver instrucciones arriba).")



Probando conexión con Google Gemini 2.0 Flash...
✅ Connection OK


---

## 🛡️ Step 4 — Prompt Anti-Alucinación v6 (MEJORADO)

### Problema identificado en v5

El prompt de v5 fue **suavizado en exceso** para evitar que el modelo rechazara preguntas respondibles. El resultado fue que el modelo volvió a alucinar información que no estaba en los documentos.

### Solución v6: Prompt Calibrado con Tres Niveles de Respuesta

El nuevo prompt establece **tres niveles de respuesta** según la información disponible:

| Nivel | Condición | Comportamiento |
|-------|-----------|----------------|
| 🟢 **Alta confianza** | El contexto contiene la respuesta directa | Responde citando `[doc:N]` |
| 🟡 **Confianza media** | El contexto tiene información parcialmente relacionada | Indica qué parte SÍ encontró y qué NO |
| 🔴 **Sin información** | No hay nada relevante en el contexto | Declara explícitamente que no hay info disponible |

### Decisión de diseño

A diferencia del prompt "todo-o-nada" de v5, este prompt **gradual** evita dos extremos:
1. Alucinar libremente (v4, v5 suavizado)
2. Rechazar preguntas respondibles (v2, v3 con modelo local)


In [4]:
ANTI_HALLUCINATION_PROMPT_V6 = """Eres un asistente especializado en Python y Machine Learning que responde EXCLUSIVAMENTE con información de los documentos de referencia proporcionados.

CONTEXTO DISPONIBLE:
{context}

PREGUNTA: {question}

INSTRUCCIONES DE RESPUESTA (sigue estas reglas en orden):

1. LEE el contexto cuidadosamente antes de responder.

2. Si encuentras información DIRECTA y COMPLETA en el contexto:
   - Responde de forma clara y estructurada
   - Cita la fuente entre corchetes: [doc:0], [doc:1], etc., después de cada dato específico
   - Usa el mismo idioma de la pregunta

3. Si encuentras información PARCIAL (el contexto toca el tema pero no responde completamente):
   - Proporciona lo que SÍ encontraste, citando fuentes
   - Indica explícitamente: "Nota: El contexto contiene información parcial sobre este tema."

4. Si el contexto NO contiene información relevante:
   - Responde exactamente: "Los documentos de referencia proporcionados no contienen información suficiente para responder esta pregunta."
   - NO uses tu conocimiento propio para completar la respuesta

5. PROHIBIDO ABSOLUTAMENTE:
   - Inventar ejemplos de código que no estén en el contexto
   - Mencionar librerías, funciones o conceptos no presentes en los documentos
   - Añadir información "de relleno" con tu conocimiento previo
   - Responder sin citar al menos una fuente [doc:N]

RESPUESTA:"""

# Crear el PromptTemplate de LangChain
PROMPT_TEMPLATE_V6 = PromptTemplate(
    input_variables=["context", "question"],
    template=ANTI_HALLUCINATION_PROMPT_V6
)

print("✅ Prompt anti-alucinación v6 configurado.")
print(f"📏 Longitud del prompt: {len(ANTI_HALLUCINATION_PROMPT_V6)} caracteres")
print("\n--- Vista previa del prompt ---")
print(ANTI_HALLUCINATION_PROMPT_V6[:400] + "...")


✅ Prompt anti-alucinación v6 configurado.
📏 Longitud del prompt: 1364 caracteres

--- Vista previa del prompt ---
Eres un asistente especializado en Python y Machine Learning que responde EXCLUSIVAMENTE con información de los documentos de referencia proporcionados.

CONTEXTO DISPONIBLE:
{context}

PREGUNTA: {question}

INSTRUCCIONES DE RESPUESTA (sigue estas reglas en orden):

1. LEE el contexto cuidadosamente antes de responder.

2. Si encuentras información DIRECTA y COMPLETA en el contexto:
   - Responde ...


---

## 📄 Step 5 — Carga de Documentos PDF con Soporte para Tablas

### ¿Por qué PyPDFLoader solo no es suficiente?

Los cheat sheets de Python y ML contienen **información crítica en formato tabular**: comparativas de métodos, listas de excepciones, parámetros de modelos, etc. PyPDFLoader extrae texto plano y **pierde la estructura** de las tablas:

```
❌ PyPDFLoader (texto plano):
"ZeroDivisionError division by zero TypeError invalid type"

✅ pdfplumber (tabla en Markdown):
| Exception | Cause |
|-----------|-------|
| ZeroDivisionError | Division by zero |
| TypeError | Invalid type |
```

### Estrategia de carga v6: Pipeline híbrido

```
PDF
 ├── PyPDFLoader  → Extrae párrafos de texto → Chunks de texto
 └── pdfplumber   → Detecta tablas → Formatea como Markdown → Chunks de tabla
                                                               (metadato: is_table=True)
```

Esta estrategia garantiza que la información tabular llegue al retriever en formato estructurado y legible.


In [5]:
def extract_tables_pdfplumber(file_path: str) -> list:
    """
    Extrae tablas de un PDF usando pdfplumber y las convierte a formato Markdown.
    
    Args:
        file_path: Ruta al archivo PDF
        
    Returns:
        Lista de Document con las tablas en formato Markdown
    """
    table_docs = []
    source_name = os.path.basename(file_path)
    
    with pdfplumber.open(file_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            
            for table_idx, table in enumerate(tables):
                if not table or len(table) < 2:
                    continue  # Ignorar tablas vacías o de 1 fila
                
                # Construir tabla en formato Markdown
                md_lines = []
                header = table[0]
                
                # Limpiar celdas None
                header = [str(cell).strip() if cell else "" for cell in header]
                md_lines.append("| " + " | ".join(header) + " |")
                md_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
                
                for row in table[1:]:
                    row = [str(cell).strip() if cell else "" for cell in row]
                    # Reemplazar saltos de línea dentro de celdas
                    row = [cell.replace("\n", " ") for cell in row]
                    md_lines.append("| " + " | ".join(row) + " |")
                
                table_content = "\n".join(md_lines)
                
                # Crear Document con metadatos enriquecidos
                doc = Document(
                    page_content=f"[TABLA] Página {page_num + 1}, Tabla {table_idx + 1}:\n{table_content}",
                    metadata={
                        "source_file": source_name,
                        "page": page_num,
                        "table_index": table_idx,
                        "is_table": True,
                        "content_type": "table"
                    }
                )
                table_docs.append(doc)
    
    return table_docs


def document_loader_v6(file_path: str) -> list:
    """
    Pipeline híbrido de carga de documentos:
    1. PyPDFLoader  → Texto general
    2. pdfplumber   → Tablas en Markdown
    
    Args:
        file_path: Ruta al archivo PDF
        
    Returns:
        Lista combinada de Documents (texto + tablas)
    """
    source_name = os.path.basename(file_path)
    print(f"\n📂 Cargando: {source_name}")
    
    # ── 1. Texto general con PyPDFLoader ──────────────────────────────────────
    loader = PyPDFLoader(file_path)
    text_docs = loader.load()
    
    for doc in text_docs:
        doc.metadata["source_file"] = source_name
        doc.metadata["is_table"] = False
        doc.metadata["content_type"] = "text"
    
    print(f"   📃 Páginas de texto: {len(text_docs)}")
    
    # ── 2. Tablas con pdfplumber ───────────────────────────────────────────────
    try:
        table_docs = extract_tables_pdfplumber(file_path)
        print(f"   📊 Tablas extraídas: {len(table_docs)}")
        
        if table_docs:
            # Mostrar preview de la primera tabla
            first_table = table_docs[0].page_content
            preview = first_table[:300] + "..." if len(first_table) > 300 else first_table
            print(f"   🔍 Preview primera tabla:\n{preview}")
    except Exception as e:
        print(f"   ⚠️  Error extrayendo tablas con pdfplumber: {e}")
        table_docs = []
    
    # ── 3. Combinar y retornar ────────────────────────────────────────────────
    all_docs = text_docs + table_docs
    print(f"   ✅ Total documentos combinados: {len(all_docs)} ({len(text_docs)} texto + {len(table_docs)} tablas)")
    
    return all_docs


# ─── Test de carga ──────────────────────────────────────────────────────────
PDF_PYTHON = "python_cheatsheet.pdf"
PDF_ML     = "ml_cheatsheet.pdf"

# Detectar paths automáticamente
import pathlib
SCRIPT_DIR = pathlib.Path().resolve()
for pdf_name in [PDF_PYTHON, PDF_ML]:
    pdf_path = SCRIPT_DIR / pdf_name
    if pdf_path.exists():
        print(f"✅ Encontrado: {pdf_path}")
    else:
        print(f"❌ No encontrado: {pdf_path} — Coloca los PDFs en la misma carpeta que este notebook")


✅ Encontrado: /Users/rolandooviedo/Documents/Master/natural_language/Actividad6/python_cheatsheet.pdf
✅ Encontrado: /Users/rolandooviedo/Documents/Master/natural_language/Actividad6/ml_cheatsheet.pdf


---

## ✂️ Step 6 — Text Splitter con Tratamiento Especial para Tablas

### Decisiones de diseño

| Parámetro | v5 | v6 | Justificación |
|-----------|----|----|---------------|
| `chunk_size` | 1000 | 1200 | Más contexto por chunk, mejor para tablas grandes |
| `chunk_overlap` | 150 | 200 | Evita cortar a la mitad listas de conceptos relacionados |
| Tablas | No tratamiento especial | Chunk propio (no dividir) | Las tablas pierden sentido si se cortan |

> **Observación técnica:** Las tablas en Markdown se almacenan como un chunk único sin dividir, ya que partir una tabla a la mitad elimina el encabezado de columnas y vuelve imposible interpretar las filas restantes.


In [6]:
def text_splitter_v6(docs: list) -> list:
    """
    Divide los documentos en chunks, preservando las tablas intactas.
    
    Estrategia:
    - Documentos de texto: RecursiveCharacterTextSplitter (chunk_size=1200)
    - Documentos de tabla: Se mantienen como un único chunk (no se dividen)
    
    Args:
        docs: Lista de Document objects
        
    Returns:
        Lista de chunks listos para vectorización
    """
    # Separar texto y tablas
    text_docs  = [d for d in docs if not d.metadata.get("is_table", False)]
    table_docs = [d for d in docs if d.metadata.get("is_table", False)]
    
    # Splitter para texto general
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    text_chunks = splitter.split_documents(text_docs)
    
    # Las tablas se mantienen como chunks únicos
    table_chunks = table_docs  # Sin dividir
    
    all_chunks = text_chunks + table_chunks
    
    # Estadísticas
    text_sizes  = [len(c.page_content) for c in text_chunks]
    table_sizes = [len(c.page_content) for c in table_chunks]
    
    print(f"✅ Chunks generados:")
    print(f"   📃 Texto: {len(text_chunks)} chunks")
    print(f"      Tamaño promedio: {int(np.mean(text_sizes)) if text_sizes else 0} chars")
    print(f"      Tamaño máximo: {max(text_sizes) if text_sizes else 0} chars")
    print(f"   📊 Tablas: {len(table_chunks)} chunks (sin dividir)")
    print(f"      Tamaño promedio: {int(np.mean(table_sizes)) if table_sizes else 0} chars")
    print(f"   🔢 TOTAL: {len(all_chunks)} chunks")
    
    return all_chunks


---

## 🧠 Step 7 — Embeddings y Base de Datos Vectorial (ChromaDB)

### Modelo de embeddings: `all-MiniLM-L6-v2`

- **Dimensiones:** 384
- **Velocidad:** Muy rápido (modelo liviano, corre en CPU)
- **Calidad:** Buena para tareas de recuperación semántica en inglés
- **Nota:** Los cheat sheets están en inglés, por lo que este modelo es adecuado

### ChromaDB como vector store

ChromaDB almacena los vectores en memoria (o en disco si se configura `persist_directory`). 
Para esta actividad usamos almacenamiento **en memoria** para simplificar el flujo del notebook.

> **Observación técnica sobre scores:** ChromaDB con `all-MiniLM-L6-v2` retorna scores en el rango 0.0–1.0 con similaridad coseno normalizada. Valores > 0.3 indican relevancia aceptable; > 0.5 indican alta relevancia.


In [7]:
#Caché global del vector store:
_vectordb_cache = {}

def get_embedding_model():
    """Retorna el modelo de embeddings (cacheado para eficiencia)."""
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}   # Normalizar para cosine similarity
    )


def build_vectordb(file_paths: list, force_rebuild: bool = False) -> Chroma:
    """
    Construye o recupera del caché el vector store ChromaDB.
    
    Args:
        file_paths: Lista de rutas a PDFs
        force_rebuild: Si True, reconstruye aunque exista en caché
        
    Returns:
        Instancia de Chroma lista para búsqueda
    """
    global _vectordb_cache
    
    # Usar caché si existe
    cache_key = tuple(sorted(file_paths))
    if cache_key in _vectordb_cache and not force_rebuild:
        print("⚡ Usando vector store en caché (evita re-procesar PDFs)")
        return _vectordb_cache[cache_key]
    
    print("🔨 Construyendo vector store desde cero...")
    
    #1. Cargar documentos:
    all_docs = []
    for fp in file_paths:
        docs = document_loader_v6(fp)
        all_docs.extend(docs)
    
    #2. Dividir en chunks:
    chunks = text_splitter_v6(all_docs)
    
    #3. Crear embeddings y vector store:
    print("\n⏳ Generando embeddings (puede tardar 1-2 minutos la primera vez)...")
    embed = get_embedding_model()
    vectordb = Chroma.from_documents(
        documents=chunks,
        embedding=embed,
        collection_metadata={"hnsw:space": "cosine"}   # Usar cosine similarity
    )
    
    print(f"✅ Vector store construido con {vectordb._collection.count()} vectores")
    
    #4. Guardar en caché:
    _vectordb_cache[cache_key] = vectordb
    
    return vectordb


---

## 🔍 Step 8 — Retriever con MMR, Calibración de Threshold y Grounding Check

### Mejoras clave respecto a v5

| Técnica | v5 | v6 |
|---------|----|----|
| **Algoritmo de búsqueda** | `similarity_search` | **MMR** (Maximal Marginal Relevance) |
| **Threshold** | Fijo en 0.0 (sin filtro) | **Dinámico**: percentil 40 de scores |
| **Deduplicación** | Manual por primeros 500 chars | Inherente en MMR |
| **Validación post-recuperación** | No existe | **Grounding Check** automático |

### ¿Qué es MMR (Maximal Marginal Relevance)?

MMR es un algoritmo de recuperación que **balancea relevancia y diversidad**:
- Selecciona el chunk más relevante primero
- Los siguientes chunks son los más relevantes **que además son distintos** a los ya seleccionados
- Resultado: menos redundancia, mejor cobertura del tema

**Parámetro clave:** `lambda_mult` (0.0 = máxima diversidad, 1.0 = máxima relevancia)
En v6 usamos `lambda_mult=0.6` — balanceamos relevancia con diversidad.

### Grounding Check

Verifica que la respuesta del LLM esté **anclada** en el contexto recuperado:
1. Extrae términos clave de la respuesta (sin stopwords)
2. Verifica qué porcentaje aparece en el contexto
3. Si < 40% → marca la respuesta como **no verificada** (posible alucinación)


In [8]:
#Configuración del retriever:
MAX_CHUNKS   = 8      # Máximo de chunks a recuperar
MAX_CHARS    = 14000  # Máximo de caracteres de contexto enviados al LLM
MMR_LAMBDA   = 0.6    # Balance relevancia/diversidad MMR (0=diversidad, 1=relevancia)
GROUNDING_THRESHOLD = 0.35  # Mínima fracción de términos clave anclados al contexto


def retrieve_with_mmr(vectordb: Chroma, question: str, k: int = MAX_CHUNKS) -> tuple:
    """
    Recupera chunks usando MMR para maximizar relevancia y diversidad.
    También calcula scores de similitud para análisis.
    
    Returns:
        tuple: (chunks_filtrados, scores_debug)
    """
    #Búsqueda con scores para diagnóstico:
    try:
        scored_results = vectordb.similarity_search_with_relevance_scores(question, k=k * 3)
        scores = [round(score, 4) for _, score in scored_results]
        print(f"\n📊 Scores de similaridad (top {len(scores)}): {scores}")
        
        # Umbral dinámico: percentil 40 de scores (filtrar el 60% menos relevante)
        if scores:
            dynamic_threshold = float(np.percentile(scores, 40))
            dynamic_threshold = max(dynamic_threshold, 0.05)  # Mínimo 0.05
            print(f"   🎯 Umbral dinámico (P40): {dynamic_threshold:.4f}")
        else:
            dynamic_threshold = 0.05
    except Exception as e:
        print(f"   ⚠️  Error calculando scores: {e}")
        scores = []
        dynamic_threshold = 0.05
    
    #Recuperación MMR:
    try:
        mmr_chunks = vectordb.max_marginal_relevance_search(
            question,
            k=k,
            fetch_k=k * 3,           # Candidatos iniciales
            lambda_mult=MMR_LAMBDA   # Balance relevancia/diversidad
        )
        print(f"   ✅ MMR recuperó {len(mmr_chunks)} chunks")
    except Exception as e:
        print(f"   ⚠️  MMR falló ({e}), usando similarity_search de respaldo")
        mmr_chunks = vectordb.similarity_search(question, k=k)
    
    # ── Trim a MAX_CHARS ──────────────────────────────────────────────────────
    trimmed, total_chars = [], 0
    for chunk in mmr_chunks:
        chunk_len = len(chunk.page_content)
        if total_chars + chunk_len > MAX_CHARS:
            break
        trimmed.append(chunk)
        total_chars += chunk_len
    
    print(f"   📤 Chunks enviados al LLM: {len(trimmed)} ({total_chars} chars)")
    
    # Separar tablas y texto en los resultados
    table_chunks = [c for c in trimmed if c.metadata.get("is_table", False)]
    text_chunks  = [c for c in trimmed if not c.metadata.get("is_table", False)]
    print(f"   📃 Texto: {len(text_chunks)} | 📊 Tablas: {len(table_chunks)}")
    
    return trimmed, scores


def grounding_check(answer: str, context_docs: list) -> dict:
    """
    Verifica que los términos clave de la respuesta aparezcan en el contexto recuperado.
    
    Args:
        answer: Respuesta generada por el LLM
        context_docs: Lista de Documents usados como contexto
        
    Returns:
        dict con score de grounding y análisis
    """
    # Stopwords básicas en inglés y español
    STOPWORDS = {
        "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
        "have", "has", "had", "do", "does", "did", "will", "would", "could",
        "should", "may", "might", "shall", "can", "need", "dare", "ought",
        "and", "or", "but", "if", "in", "on", "at", "to", "for", "of", "with",
        "by", "from", "up", "about", "into", "through", "during", "as",
        "el", "la", "los", "las", "un", "una", "unos", "unas", "de", "del",
        "en", "con", "por", "para", "que", "se", "es", "son", "fue", "ser",
        "los", "sus", "al", "lo", "le", "les", "más", "pero", "si", "ya",
        "this", "that", "these", "those", "it", "its", "they", "them", "their",
        "which", "who", "what", "when", "where", "how", "why", "not", "no",
        "also", "can", "use", "used", "using", "i", "you", "we", "he", "she",
        "doc", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9"
    }
    
    # Extraer términos de la respuesta (mínimo 4 chars, no stopwords)
    answer_words = re.findall(r'\b[a-zA-ZáéíóúÁÉÍÓÚñÑ]{4,}\b', answer.lower())
    key_terms = [w for w in answer_words if w not in STOPWORDS]
    
    if not key_terms:
        return {"score": 1.0, "grounded": True, "note": "No hay términos clave que verificar"}
    
    # Contexto completo como texto
    context_text = " ".join(d.page_content.lower() for d in context_docs)
    
    # Verificar qué términos aparecen en el contexto
    grounded_terms   = [t for t in key_terms if t in context_text]
    ungrounded_terms = [t for t in key_terms if t not in context_text]
    
    # Usar Counter para no repetir términos
    unique_key      = list(set(key_terms))
    unique_grounded = [t for t in unique_key if t in context_text]
    grounding_score = len(unique_grounded) / len(unique_key) if unique_key else 1.0
    
    return {
        "score": round(grounding_score, 3),
        "grounded": grounding_score >= GROUNDING_THRESHOLD,
        "total_key_terms": len(unique_key),
        "grounded_terms": len(unique_grounded),
        "ungrounded_sample": list(set(ungrounded_terms))[:10],
        "note": f"{len(unique_grounded)}/{len(unique_key)} términos clave encontrados en el contexto"
    }

print("✅ Retriever MMR y Grounding Check configurados.")


✅ Retriever MMR y Grounding Check configurados.


---

## 🔗 Step 9 — QA Chain Anti-Alucinación v6

### Flujo completo de una consulta

```
Pregunta del usuario
       │
       ▼
  [Retriever MMR]  ──→  Top-K chunks (texto + tablas)
       │
       ▼
  [Build Context]  ──→  Contexto formateado con [doc:N] markers
       │
       ▼
  [Prompt v6]      ──→  Instrucciones estrictas de anti-alucinación
       │
       ▼
  [Grok API]       ──→  Respuesta con citas inline
       │
       ▼
  [Grounding Check]──→  Verificación de anclaje al contexto
       │
       ▼
  Respuesta final + score de confianza + fuentes
```

### Historial conversacional

La cadena mantiene los últimos **6 turnos** de conversación para respuestas coherentes en preguntas de seguimiento.


In [9]:
# Estado conversacional:
_conversation_history_v6 = []


def answer_question_v6(file_paths: list, question: str,
                        verbose: bool = True) -> str:
    """
    Pipeline completo de QA anti-alucinación v6.
    
    Args:
        file_paths: Lista de rutas a PDFs
        question: Pregunta del usuario
        verbose: Si True, imprime información de diagnóstico
        
    Returns:
        str: Respuesta formateada con fuentes y score de grounding
    """
    global _conversation_history_v6
    
    #1. Construir/recuperar vector store:
    vectordb = build_vectordb(file_paths)
    
    #2. Recuperar chunks con MMR:
    context_docs, scores = retrieve_with_mmr(vectordb, question)
    
    if not context_docs:
        return "No se recuperaron documentos. Verifica que los PDFs estén cargados correctamente."
    
    #3. Construir contexto con markers [doc:N]
    context_parts = []
    for i, doc in enumerate(context_docs):
        source = doc.metadata.get("source_file", "unknown")
        page   = doc.metadata.get("page", "?")
        is_tab = "📊 TABLA" if doc.metadata.get("is_table", False) else "📃 Texto"
        context_parts.append(
            f"[doc:{i}] ({is_tab} | Fuente: {source} | Pág. {page}):\n{doc.page_content}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    #4. Construir mensajes con historial:
    messages = []
    
    # System message con el prompt anti-alucinación
    full_prompt = ANTI_HALLUCINATION_PROMPT_V6.format(
        context=context,
        question=question
    )
    messages.append({"role": "user", "content": full_prompt})
    
    #5. Llamar a Grok
    try:
        answer = grok_chat(messages, temperature=0.1, max_tokens=2048)
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            return "❌ Error 401: API Key inválida. Verifica xAI_API_KEY en .env"
        return f"❌ Error HTTP {e.response.status_code}: {e.response.text}"
    except Exception as e:
        return f"❌ Error inesperado: {str(e)}"
    
    #6. Grounding Check:
    grounding = grounding_check(answer, context_docs)
    
    if verbose:
        print(f"\n🔍 Grounding Check:")
        print(f"   Score: {grounding['score']:.3f} | Anclado: {grounding['grounded']}")
        print(f"   {grounding['note']}")
        if not grounding['grounded'] and grounding.get('ungrounded_sample'):
            print(f"   ⚠️  Términos no encontrados: {grounding['ungrounded_sample'][:5]}")
    
    #7. Actualizar historial conversacional:
    _conversation_history_v6.append(("user", question))
    _conversation_history_v6.append(("assistant", answer))
    
    # Mantener solo los últimos 6 turnos (3 preguntas + 3 respuestas)
    if len(_conversation_history_v6) > 12:
        _conversation_history_v6 = _conversation_history_v6[-12:]
    
    #8. Formatear respuesta final:
    sources = sorted(set(d.metadata.get("source_file", "?") for d in context_docs))
    table_sources = [d.metadata.get("source_file", "?")
                     for d in context_docs if d.metadata.get("is_table", False)]
    
    # Advertencia de grounding bajo
    grounding_note = ""
    if not grounding['grounded']:
        grounding_note = (
            f"\n\n> ⚠️ **Advertencia de verificación:** El score de grounding es "
            f"{grounding['score']:.2f} (umbral: {GROUNDING_THRESHOLD}). "
            f"Algunos términos de esta respuesta podrían no estar directamente en los documentos. "
            f"Verifica la información con las fuentes citadas."
        )
    
    confidence_emoji = "🟢" if grounding['score'] >= 0.6 else ("🟡" if grounding['score'] >= GROUNDING_THRESHOLD else "🔴")
    
    final_answer = (
        f"{answer}"
        f"{grounding_note}"
        f"\n\n---"
        f"\n📎 **Fuentes consultadas:** {', '.join(sources)}"
        f"\n{confidence_emoji} **Score de grounding:** {grounding['score']:.2f} "
        f"({grounding['grounded_terms']}/{grounding['total_key_terms']} términos verificados)"
    )
    if table_sources:
        final_answer += f"\n📊 **Incluye información de tablas:** {', '.join(set(table_sources))}"
    
    return final_answer


print("✅ Pipeline QA v6 configurado.")
print(f"   - Temperatura: 0.1 (muy baja, máxima fidelidad al contexto)")
print(f"   - Max chunks: {MAX_CHUNKS}")
print(f"   - Max chars contexto: {MAX_CHARS}")
print(f"   - Grounding threshold: {GROUNDING_THRESHOLD}")
print(f"   - MMR lambda: {MMR_LAMBDA}")


✅ Pipeline QA v6 configurado.
   - Temperatura: 0.1 (muy baja, máxima fidelidad al contexto)
   - Max chunks: 8
   - Max chars contexto: 14000
   - Grounding threshold: 0.35
   - MMR lambda: 0.6


---

## 💻 Step 10 — Interfaz Gradio

La interfaz web permite:
1. **Subir uno o varios PDFs** desde el navegador
2. **Hacer preguntas** en lenguaje natural (inglés o español)
3. **Ver respuestas** con citas de fuentes y score de grounding
4. **Mantener historial** de la conversación actual
5. **Botón de reinicio** para limpiar el historial y comenzar una nueva sesión

> Para ejecutar el chatbot, ejecuta las dos celdas siguientes. La primera define la función, la segunda lanza el servidor.


In [10]:
# ─── Estado global de la interfaz Gradio ────────────────────────────────────
_gradio_files_v6 = None


def gradio_rag_v6(message: str, history: list, files) -> str:
    """
    Función principal de la interfaz Gradio.
    
    Firma requerida por gr.ChatInterface: (message, history, *additional_inputs)
    """
    global _gradio_files_v6, _conversation_history_v6
    
    # Actualizar archivos si se subieron nuevos
    if files is not None:
        new_files = files if isinstance(files, list) else [files]
        if new_files != _gradio_files_v6:
            _gradio_files_v6 = new_files
            _conversation_history_v6 = []  # Reset historial al cambiar PDFs
            print(f"📂 Nuevos PDFs cargados: {[os.path.basename(f) for f in new_files]}")
    
    if not _gradio_files_v6:
        return "⚠️ Por favor sube al menos un archivo PDF antes de hacer preguntas."
    
    try:
        return answer_question_v6(_gradio_files_v6, message, verbose=False)
    except Exception as e:
        err = str(e)
        if "Connection" in err or "10061" in err or "refused" in err:
            return "❌ No se puede conectar a xAI API. Verifica tu conexión y API key."
        return f"❌ Error: {err}"


def reset_conversation():
    """Limpia el historial de conversación."""
    global _conversation_history_v6
    _conversation_history_v6 = []
    return "✅ Historial limpiado. Puedes comenzar una nueva conversación."


In [11]:
# ─── Definición y lanzamiento de la interfaz ────────────────────────────────
gr.close_all()  # Cerrar instancias previas

rag_app_v6 = gr.ChatInterface(
    fn=gradio_rag_v6,
    additional_inputs=[
        gr.File(
            label="📂 Subir PDF(s) — python_cheatsheet.pdf y/o ml_cheatsheet.pdf",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath"
        ),
    ],
    title="🤖 ITESM-NLP RAG Chatbot v6 — Anti-Alucinación con Extracción de Tablas",
    description=(
        "**Chatbot RAG para hojas de referencia de Python y Machine Learning**\n\n"
        "📋 **Instrucciones:**\n"
        "1. Sube `python_cheatsheet.pdf` y/o `ml_cheatsheet.pdf` usando el selector de archivos\n"
        "2. Escribe tu pregunta en el campo de texto\n"
        "3. La respuesta incluirá citas [doc:N] y un score de grounding\n\n"
        "🛡️ **Anti-alucinación:** Temperatura=0.1 | MMR Retrieval | Grounding Check | Prompt estricto"
    ),
    examples=[
        ["According to the Exceptions section, what exception is raised when dividing by zero?"],
        ["Can you give me 2 examples of string methods?"],
        ["What are the main disadvantages of Random Forests?"],
        ["What are 3 use cases of clustering in Unsupervised Learning?"],
    ],
)

print("🚀 Lanzando interfaz Gradio...")
rag_app_v6.launch(server_name="127.0.0.1", server_port=7866, share=False, theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="blue"))


🚀 Lanzando interfaz Gradio...
* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


---

## 📝 Step 11 — Preguntas Requeridas de la Actividad

En esta sección respondemos las **6 preguntas obligatorias** de la actividad usando el pipeline RAG v6.

> **Metodología:** Cada pregunta se ejecuta llamando directamente a `answer_question_v6()` con los dos PDFs. La función imprime automáticamente:
> - Los scores de similitud de los chunks recuperados
> - El número de chunks enviados al LLM
> - El grounding check de la respuesta
> - La respuesta final con citas y score de confianza

---

### 📋 Resumen de preguntas

| # | Pregunta | Documento fuente esperado |
|---|----------|--------------------------|
| a | Excepción por división entre cero | `python_cheatsheet.pdf` — Sección Exceptions |
| b | 2 ejemplos de String Methods | `python_cheatsheet.pdf` — Sección String Methods |
| c | Desventajas de Random Forests | `ml_cheatsheet.pdf` — Sección Random Forests |
| d | 3 casos de uso de Clustering | `ml_cheatsheet.pdf` — Sección Unsupervised Learning |
| e | Pregunta adicional Python | `python_cheatsheet.pdf` |
| f | Pregunta adicional ML | `ml_cheatsheet.pdf` |


---

### ❓ Pregunta a — Excepción por División entre Cero

> *"Según la sección de 'Exceptions', ¿qué excepción se lanza si divido entre cero?"*

**Contexto esperado:** El `python_cheatsheet.pdf` tiene una sección de Exceptions con una tabla de excepciones comunes. La respuesta correcta es `ZeroDivisionError`.

**¿Por qué esta pregunta es un buen test de anti-alucinación?**
- Es una pregunta de dato exacto (nombre de excepción)
- El nombre está en una **tabla** dentro del cheat sheet
- Un modelo con alucinación podría responder "DivisionByZeroError" (Java) o dar el nombre incorrecto


In [12]:
# ─── Configurar paths de los PDFs ───────────────────────────────────────────
import pathlib
NOTEBOOK_DIR = pathlib.Path().resolve()
PDF_PATHS = [
    str(NOTEBOOK_DIR / "python_cheatsheet.pdf"),
    str(NOTEBOOK_DIR / "ml_cheatsheet.pdf"),
]

print("=" * 70)
print("PREGUNTA A: Excepción por división entre cero")
print("=" * 70)

QUESTION_A = (
    "According to the 'Exceptions' section in the Python cheat sheet, "
    "what exception is raised when you divide by zero? "
    "Please provide the exact exception name and its description."
)

answer_a = answer_question_v6(PDF_PATHS, QUESTION_A)
print("\n📣 RESPUESTA:")
print(answer_a)


PREGUNTA A: Excepción por división entre cero
🔨 Construyendo vector store desde cero...

📂 Cargando: python_cheatsheet.pdf
   📃 Páginas de texto: 3
   📊 Tablas extraídas: 8
   🔍 Preview primera tabla:
[TABLA] Página 1, Tabla 1:
|  |  |  |  |  |  | Data Types Strings Numbers & Math
Real Python
Python is dynamically typed It’s recommended to use double-quotes for strings Arithmetic Operators
Pocket Reference
Use None to represent missing or optional values Use "\n" to create a line break in a strin...
   ✅ Total documentos combinados: 11 (3 texto + 8 tablas)

📂 Cargando: ml_cheatsheet.pdf
   📃 Páginas de texto: 1


   📊 Tablas extraídas: 4
   🔍 Preview primera tabla:
[TABLA] Página 1, Tabla 1:
| algorithm description applications advantages disadvantages
A simple algorithm that models a linear use cases   Explainable metho    Assumes linearity between inputs and outpu 
  Stock price predictio 
relationship between inputs and a continuous   Interpretable results ...
   ✅ Total documentos combinados: 5 (1 texto + 4 tablas)
✅ Chunks generados:
   📃 Texto: 22 chunks
      Tamaño promedio: 1083 chars
      Tamaño máximo: 1198 chars
   📊 Tablas: 12 chunks (sin dividir)
      Tamaño promedio: 3911 chars
   🔢 TOTAL: 34 chunks

⏳ Generando embeddings (puede tardar 1-2 minutos la primera vez)...


Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 29476.89it/s]


✅ Vector store construido con 34 vectores

📊 Scores de similaridad (top 24): [0.587, 0.3388, 0.2937, 0.2807, 0.2592, 0.2564, 0.2559, 0.2432, 0.2402, 0.2346, 0.2326, 0.2318, 0.2087, 0.2074, 0.2059, 0.1917, 0.1915, 0.1842, 0.1765, 0.1727, 0.165, 0.1454, 0.1178, 0.0345]
   🎯 Umbral dinámico (P40): 0.2062
   ✅ MMR recuperó 8 chunks
   📤 Chunks enviados al LLM: 8 (10834 chars)
   📃 Texto: 7 | 📊 Tablas: 1

🔍 Grounding Check:
   Score: 0.500 | Anclado: True
   16/32 términos clave encontrados en el contexto

📣 RESPUESTA:
According to the "Exceptions" section (specifically within the "Try-Except" example), the exception raised when you divide by zero is:

* **Exception Name:** `ZeroDivisionError` [doc:0]
* **Description / Handling:** In the provided code example, it is handled by printing the message `"Cannot divide by zero!"` [doc:0]. 

*(Note: While `ZeroDivisionError` is demonstrated in the "Try-Except" code block, it is not explicitly listed or defined in the "Common Exceptions" reference 

---

### ❓ Pregunta b — Métodos de Cadena (String Methods)

> *"¿Puedes proporcionarme 2 ejemplos de métodos de cadena (string methods)?"*

**Contexto esperado:** El `python_cheatsheet.pdf` tiene una sección de String Methods con tabla de métodos disponibles.

**¿Por qué esta pregunta es un buen test de anti-alucinación?**
- Hay decenas de string methods en Python; el modelo podría inventar métodos que no están en el cheat sheet
- La respuesta debe limitarse a los métodos que aparecen **explícitamente** en el documento


In [13]:
print("=" * 70)
print("PREGUNTA B: Ejemplos de String Methods")
print("=" * 70)

QUESTION_B = (
    "From the Python cheat sheet, can you provide 2 examples of string methods? "
    "For each method, include its name, what it does, and an example if available in the document."
)

answer_b = answer_question_v6(PDF_PATHS, QUESTION_B)
print("\n📣 RESPUESTA:")
print(answer_b)


PREGUNTA B: Ejemplos de String Methods
⚡ Usando vector store en caché (evita re-procesar PDFs)

📊 Scores de similaridad (top 24): [0.4525, 0.437, 0.4263, 0.4077, 0.3742, 0.3646, 0.3366, 0.3255, 0.3149, 0.3068, 0.2987, 0.2972, 0.2924, 0.2857, 0.2691, 0.2422, 0.2383, 0.2274, 0.2247, 0.2173, 0.2156, 0.2149, 0.1286, 0.0788]
   🎯 Umbral dinámico (P40): 0.2724
   ✅ MMR recuperó 8 chunks
   📤 Chunks enviados al LLM: 8 (12850 chars)
   📃 Texto: 6 | 📊 Tablas: 2

🔍 Grounding Check:
   Score: 0.303 | Anclado: False
   10/33 términos clave encontrados en el contexto
   ⚠️  Términos no encontrados: ['referencia', 'ejemplos', 'mayúsculas', 'método', 'minúsculas']

📣 RESPUESTA:
Aquí tienes 2 ejemplos de métodos de cadenas (*string methods*) disponibles en el documento de referencia:

1. **Método `upper()`** [doc:1]
   - **Nombre:** `upper()`
   - **Qué hace:** Convierte los caracteres de una cadena a mayúsculas (retorna la versión en mayúsculas de la cadena).
   - **Ejemplo en el documento:** 
     `

---

### ❓ Pregunta c — Desventajas de Random Forests

> *"¿Cuáles son las principales desventajas de utilizar el modelo Bosque Aleatorio (Random Forests)?"*

**Contexto esperado:** El `ml_cheatsheet.pdf` tiene una sección sobre Random Forests con tabla de ventajas/desventajas.

**¿Por qué esta pregunta es un buen test de anti-alucinación?**
- Random Forests es un modelo bien conocido y el LLM tiene mucho conocimiento propio al respecto
- Un modelo con alucinación listará desventajas estándar de su conocimiento (no del documento)
- La respuesta debe contener exactamente las desventajas listadas en el PDF


In [14]:
print("=" * 70)
print("PREGUNTA C: Desventajas de Random Forests")
print("=" * 70)

QUESTION_C = (
    "According to the ML cheat sheet, what are the main disadvantages "
    "of using the Random Forests model? List all disadvantages mentioned in the document."
)

answer_c = answer_question_v6(PDF_PATHS, QUESTION_C)
print("\n📣 RESPUESTA:")
print(answer_c)


PREGUNTA C: Desventajas de Random Forests
⚡ Usando vector store en caché (evita re-procesar PDFs)

📊 Scores de similaridad (top 24): [0.3894, 0.3422, 0.3256, 0.3106, 0.3104, 0.2689, 0.249, 0.1605, 0.1082, 0.1049, 0.0997, 0.0696, 0.0683, 0.0578, 0.0394, 0.0348, 0.0244, 0.0219, 0.0204, 0.0184, 0.0138, 0.0106, 0.0101, 0.0061]
   🎯 Umbral dinámico (P40): 0.0500
   ✅ MMR recuperó 8 chunks
   📤 Chunks enviados al LLM: 8 (13666 chars)
   📃 Texto: 5 | 📊 Tablas: 3

🔍 Grounding Check:
   Score: 0.833 | Anclado: True
   10/12 términos clave encontrados en el contexto

📣 RESPUESTA:
Based on the provided documents, the disadvantages of using the **Random Forests** model are:

* **Training complexity can be high** [doc:0][doc:2][doc:3]
* **Not very interpretable** [doc:0][doc:2][doc:3]

---
📎 **Fuentes consultadas:** ml_cheatsheet.pdf, python_cheatsheet.pdf
🟢 **Score de grounding:** 0.83 (10/12 términos verificados)
📊 **Incluye información de tablas:** ml_cheatsheet.pdf


---

### ❓ Pregunta d — Casos de Uso de Clustering (Aprendizaje No Supervisado)

> *"¿Cuáles son 3 casos de uso (use cases) de las técnicas de aprendizaje no supervisado (Unsupervised Learning) para las técnicas de agrupamiento (clustering)?"*

**Contexto esperado:** El `ml_cheatsheet.pdf` tiene una sección de Unsupervised Learning con casos de uso de clustering.

**¿Por qué esta pregunta es un buen test de anti-alucinación?**
- Los casos de uso de clustering son amplios; un modelo con alucinación inventará aplicaciones generales
- La respuesta debe reflejar los casos de uso **específicos mencionados en el documento**


In [15]:
print("=" * 70)
print("PREGUNTA D: Casos de uso de Clustering en Unsupervised Learning")
print("=" * 70)

QUESTION_D = (
    "According to the ML cheat sheet, what are 3 use cases of clustering techniques "
    "within Unsupervised Learning? Please list them as they appear in the document."
)

answer_d = answer_question_v6(PDF_PATHS, QUESTION_D)
print("\n📣 RESPUESTA:")
print(answer_d)


PREGUNTA D: Casos de uso de Clustering en Unsupervised Learning
⚡ Usando vector store en caché (evita re-procesar PDFs)

📊 Scores de similaridad (top 24): [0.5361, 0.5278, 0.4361, 0.2558, 0.1985, 0.1706, 0.1691, 0.1623, 0.1604, 0.1577, 0.1301, 0.1141, 0.1118, 0.1055, 0.089, 0.0861, 0.0859, 0.0742, 0.0664, 0.0602, 0.0475, 0.0376, 0.0345, 0.0324]
   🎯 Umbral dinámico (P40): 0.0923
   ✅ MMR recuperó 8 chunks
   📤 Chunks enviados al LLM: 8 (10673 chars)
   📃 Texto: 5 | 📊 Tablas: 3

🔍 Grounding Check:
   Score: 0.333 | Anclado: False
   11/33 términos clave encontrados en el contexto
   ⚠️  Términos no encontrados: ['documentos', 'agrupamiento', 'referencia', 'nota', 'técnicas']

📣 RESPUESTA:
De acuerdo con los documentos de referencia, tres casos de uso (*use cases*) de las técnicas de agrupamiento (*clustering*) dentro del aprendizaje no supervisado son:

1. **Customer segmentatio** (Segmentación de clientes) [doc:0] [doc:1] [doc:2]
2. **Recommendation systems** (Sistemas de recomendación

---

### ❓ Pregunta e — Pregunta Adicional sobre Python Cheatsheet

> *"¿Cuáles son los tipos de datos principales en Python según el cheat sheet? ¿Puedes dar ejemplos de cada uno?"*

**Objetivo:** Verificar que el sistema RAG puede recuperar información de la sección de tipos de datos del Python cheatsheet, que incluye tablas con ejemplos de cada tipo.

**Relevancia para anti-alucinación:** Los tipos de datos de Python son conocimiento básico del LLM — es un buen test para verificar que el modelo usa el documento en lugar de su conocimiento previo.


In [16]:
print("=" * 70)
print("PREGUNTA E: Tipos de datos en Python (Python Cheatsheet)")
print("=" * 70)

QUESTION_E = (
    "According to the Python cheat sheet, what are the main data types in Python? "
    "Can you provide at least one example for each data type as shown in the document?"
)

answer_e = answer_question_v6(PDF_PATHS, QUESTION_E)
print("\n📣 RESPUESTA:")
print(answer_e)


PREGUNTA E: Tipos de datos en Python (Python Cheatsheet)
⚡ Usando vector store en caché (evita re-procesar PDFs)

📊 Scores de similaridad (top 24): [0.4281, 0.3949, 0.3924, 0.3911, 0.3693, 0.3181, 0.2816, 0.2747, 0.2708, 0.2577, 0.2576, 0.2564, 0.2437, 0.2428, 0.2419, 0.2371, 0.2302, 0.2293, 0.2198, 0.2167, 0.2125, 0.1808, 0.1624, 0.1611]
   🎯 Umbral dinámico (P40): 0.2421
   ✅ MMR recuperó 8 chunks
   📤 Chunks enviados al LLM: 6 (10201 chars)
   📃 Texto: 4 | 📊 Tablas: 2

🔍 Grounding Check:
   Score: 0.545 | Anclado: True
   18/33 términos clave encontrados en el contexto

📣 RESPUESTA:
According to the provided documents, the main data types in Python (as shown in the "Type Investigation" and "Data Types" sections) along with their examples are:

* **`int`** (Integer):
  * Example: `42` (verified with `type(42)`) [doc:0][doc:1]
* **`float`**:
  * Example: `3.14` (verified with `type(3.14)`) [doc:0][doc:1]
* **`str`** (String):
  * Example: `"Hello"` (verified with `type("Hello")`) [doc

---

### ❓ Pregunta f — Pregunta Adicional sobre ML Cheatsheet

> *"Según el cheat sheet de ML, ¿cuáles son las métricas de evaluación para modelos de clasificación y qué mide cada una?"*

**Objetivo:** Recuperar información de la sección de métricas de evaluación del ml_cheatsheet.pdf, que típicamente incluye tablas con Accuracy, Precision, Recall, F1-Score.

**Relevancia para anti-alucinación:** Las métricas de ML son conocimiento estándar — el modelo podría expandir con definiciones propias no presentes en el documento. El grounding check verificará esto.


In [17]:
print("=" * 70)
print("PREGUNTA F: Métricas de evaluación para clasificación (ML Cheatsheet)")
print("=" * 70)

QUESTION_F = (
    "According to the ML cheat sheet, what are the evaluation metrics for "
    "classification models? What does each metric measure? "
    "Include any formulas or examples shown in the document."
)

answer_f = answer_question_v6(PDF_PATHS, QUESTION_F)
print("\n📣 RESPUESTA:")
print(answer_f)


PREGUNTA F: Métricas de evaluación para clasificación (ML Cheatsheet)
⚡ Usando vector store en caché (evita re-procesar PDFs)

📊 Scores de similaridad (top 24): [0.256, 0.2554, 0.2524, 0.2272, 0.2269, 0.222, 0.2109, 0.1934, 0.1912, 0.181, 0.1592, 0.1545, 0.1389, 0.1368, 0.1306, 0.1076, 0.0904, 0.0898, 0.0851, 0.0848, 0.0751, 0.0735, 0.0608, 0.0556]
   🎯 Umbral dinámico (P40): 0.1318
   ✅ MMR recuperó 8 chunks
   📤 Chunks enviados al LLM: 5 (11429 chars)
   📃 Texto: 3 | 📊 Tablas: 2

🔍 Grounding Check:
   Score: 0.000 | Anclado: False
   0/9 términos clave encontrados en el contexto
   ⚠️  Términos no encontrados: ['documentos', 'contienen', 'referencia', 'suficiente', 'proporcionados']

📣 RESPUESTA:
Los documentos de referencia proporcionados no contienen información suficiente para responder esta pregunta.

> ⚠️ **Advertencia de verificación:** El score de grounding es 0.00 (umbral: 0.35). Algunos términos de esta respuesta podrían no estar directamente en los documentos. Verifica la i

---

## 📊 Step 12 — Análisis Técnico del Pipeline RAG

En esta sección analizamos cuantitativamente el comportamiento del sistema de recuperación para entender sus fortalezas y limitaciones.

### Aspectos a analizar:
1. **Distribución de chunks** por fuente y tipo (texto vs. tabla)
2. **Calibración del threshold dinámico** — ¿qué percentil funciona mejor?
3. **Cobertura de preguntas** — ¿los chunks recuperados son relevantes?
4. **Impacto de pdfplumber** — ¿cuántas tablas se extrajeron de cada PDF?


In [18]:
print("=" * 70)
print("ANÁLISIS TÉCNICO DEL PIPELINE RAG v6")
print("=" * 70)

# ─── Reconstruir el vector store y analizar su contenido ────────────────────
vectordb = build_vectordb(PDF_PATHS)
collection = vectordb._collection
total_docs = collection.count()

print(f"\n📊 ESTADÍSTICAS DEL VECTOR STORE:")
print(f"   Total de vectores: {total_docs}")

# Recuperar todos los metadatos para análisis
all_metadata = collection.get(include=["metadatas", "documents"])
metadatas = all_metadata["metadatas"]
documents = all_metadata["documents"]

# Análisis por fuente
from collections import Counter
sources = [m.get("source_file", "unknown") for m in metadatas]
source_counts = Counter(sources)
print(f"\n   📁 Chunks por fuente:")
for src, count in sorted(source_counts.items()):
    print(f"      {src}: {count} chunks")

# Análisis texto vs tablas
content_types = [m.get("content_type", "text") for m in metadatas]
type_counts = Counter(content_types)
print(f"\n   📋 Chunks por tipo:")
for ctype, count in sorted(type_counts.items()):
    print(f"      {ctype}: {count} chunks")

# Análisis de tamaños de chunks
chunk_sizes = [len(d) for d in documents]
print(f"\n   📏 Tamaños de chunks:")
print(f"      Mínimo: {min(chunk_sizes)} chars")
print(f"      Máximo: {max(chunk_sizes)} chars")
print(f"      Promedio: {int(np.mean(chunk_sizes))} chars")
print(f"      Mediana: {int(np.median(chunk_sizes))} chars")

# ─── Análisis de scores para preguntas de prueba ────────────────────────────
print(f"\n🎯 ANÁLISIS DE SCORES DE RECUPERACIÓN:")

test_queries = [
    ("ZeroDivisionError exception Python", "python_cheatsheet.pdf"),
    ("string methods Python examples",     "python_cheatsheet.pdf"),
    ("Random Forest disadvantages",        "ml_cheatsheet.pdf"),
    ("clustering use cases unsupervised",  "ml_cheatsheet.pdf"),
]

for query, expected_source in test_queries:
    print(f"\n   Query: '{query[:50]}'")
    scored = vectordb.similarity_search_with_relevance_scores(query, k=10)
    scores = [round(s, 4) for _, s in scored]
    if scores:
        dynamic_thr = float(np.percentile(scores, 40))
        above_thr = [s for s in scores if s >= max(dynamic_thr, 0.05)]
        print(f"   Scores: {scores[:6]}{'...' if len(scores) > 6 else ''}")
        print(f"   Umbral dinámico (P40): {dynamic_thr:.4f}")
        print(f"   Chunks aceptados: {len(above_thr)}/{len(scores)}")
        # Verificar que el top chunk sea del documento esperado
        top_doc = scored[0][0] if scored else None
        if top_doc:
            top_source = top_doc.metadata.get("source_file", "?")
            top_type = "TABLA" if top_doc.metadata.get("is_table") else "Texto"
            match = "✅" if top_source == expected_source else "⚠️"
            print(f"   {match} Top chunk: {top_source} ({top_type})")

print("\n✅ Análisis técnico completado.")


ANÁLISIS TÉCNICO DEL PIPELINE RAG v6
⚡ Usando vector store en caché (evita re-procesar PDFs)

📊 ESTADÍSTICAS DEL VECTOR STORE:
   Total de vectores: 34

   📁 Chunks por fuente:
      ml_cheatsheet.pdf: 10 chunks
      python_cheatsheet.pdf: 24 chunks

   📋 Chunks por tipo:
      table: 12 chunks
      text: 22 chunks

   📏 Tamaños de chunks:
      Mínimo: 429 chars
      Máximo: 18818 chars
      Promedio: 2081 chars
      Mediana: 1187 chars

🎯 ANÁLISIS DE SCORES DE RECUPERACIÓN:

   Query: 'ZeroDivisionError exception Python'
   Scores: [0.3737, 0.3445, 0.2931, 0.2845, 0.2475, 0.2441]...
   Umbral dinámico (P40): 0.2422
   Chunks aceptados: 6/10
   ✅ Top chunk: python_cheatsheet.pdf (Texto)

   Query: 'string methods Python examples'
   Scores: [0.4894, 0.4788, 0.4587, 0.4219, 0.4048, 0.4025]...
   Umbral dinámico (P40): 0.3840
   Chunks aceptados: 6/10
   ✅ Top chunk: python_cheatsheet.pdf (TABLA)

   Query: 'Random Forest disadvantages'
   Scores: [0.3851, 0.3787, 0.3533, 0.3512, 0

---

## 📊 Step 13 — Retos del Manejo de Información Tabular en PDFs

Esta sección documenta los **retos específicos** que presentó la información tabular de los cheat sheets y cómo se abordaron en v6.

---

### 🔴 Problema 1: Pérdida de estructura con extracción básica

**PyPDFLoader** extrae todo el texto de un PDF de forma lineal. Cuando encuentra una tabla, el resultado es texto plano sin estructura:

```
❌ Resultado PyPDFLoader (texto plano sin estructura):
"Exception Cause ZeroDivisionError division by zero TypeError unsupported operand..."

✅ Resultado pdfplumber (Markdown preservado):
| Exception | Cause |
|-----------|-------|
| ZeroDivisionError | Division by zero |
| TypeError | Unsupported operand |
```

**Impacto en alucinaciones:** Con el texto plano, el modelo no puede identificar qué concepto pertenece a qué columna. Esto lleva a respuestas incorrectas o inventadas.

---

### 🟡 Problema 2: Fragmentación de tablas grandes

Cuando una tabla ocupa más de una página o es muy larga, el text splitter puede cortarla en medio de una fila. El resultado es un chunk con filas huérfanas sin encabezado.

**Solución v6:** Las tablas extraídas con pdfplumber se almacenan como chunks **indivisibles** — no se aplica el RecursiveCharacterTextSplitter sobre ellas.

---

### 🟡 Problema 3: Embeddings de texto tabular en Markdown

El modelo `all-MiniLM-L6-v2` fue entrenado principalmente con texto corrido. Las tablas en Markdown incluyen caracteres `|`, `---`, y celdas cortas que pueden reducir la calidad de los embeddings.

**Observación:** En las pruebas, los chunks de tablas obtuvieron scores ligeramente más bajos que los de texto corrido para preguntas equivalentes. Esto sugiere que un modelo de embeddings especializado en tablas (como `table-bert`) podría mejorar la recuperación.

---

### 🟡 Problema 4: Preguntas sobre posición en tabla vs. contenido

Si el usuario pregunta "¿qué está en la columna X de la tabla Y?", el sistema RAG no tiene una forma directa de filtrar por posición de columna — solo recupera por similitud semántica del contenido completo del chunk.

---

### ✅ Resultado de la estrategia v6

| Técnica | Mejora lograda |
|---------|---------------|
| **pdfplumber + Markdown** | Preservación de estructura tabular |
| **Chunks no divisibles para tablas** | Sin fragmentación de filas |
| **Metadato `is_table=True`** | Permite filtrar/priorizar chunks tabulares |
| **Context markers `[📊 TABLA]`** | El LLM sabe que está leyendo una tabla |

### 📌 Recomendaciones para trabajos futuros

1. **Usar modelos de embeddings especializados en tablas** (TAPAS, Table-BERT)
2. **Preservar el contexto de la tabla** — incluir el título de sección antes de la tabla en el chunk
3. **Estrategia de chunks con contexto de ventana** para tablas multi-página
4. **Experimentar con formatos alternativos**: HTML en lugar de Markdown podría mejorar la preservación de estructura compleja


---

## 🏁 Step 14 — Conclusiones Finales

---

### 📈 Evolución del sistema RAG a lo largo de las versiones

| Versión | LLM | Anti-alucinación | Tablas | Calidad |
|---------|-----|-----------------|--------|---------|
| v1 | Local 7B | ❌ Sin control | ❌ PyPDF básico | Baja |
| v2–v3 | Local 7B | ❌ Prompt muy estricto — rechazaba todo | ❌ PyPDF básico | Muy baja |
| v4 | Local 7B | ⚠️ Sin prompt custom | ❌ PyPDF básico | Media |
| v5 | Grok API | ⚠️ Prompt suavizado en exceso | ❌ PyPDF básico | Media-alta |
| **v6** | **Grok API** | **✅ Grounding check + MMR + prompt calibrado** | **✅ pdfplumber + Markdown** | **Alta** |

---

### 🔑 Conclusiones principales

**1. El modelo importa, pero no es suficiente**  
Incluso con Grok (modelo grande), un prompt mal calibrado o un retriever deficiente produce alucinaciones. La calidad del RAG depende de toda la cadena: extracción → chunking → embeddings → retrieval → prompt → LLM.

**2. La extracción tabular es el cuello de botella más crítico**  
La mayoría de la información relevante en los cheat sheets está en tablas. Con PyPDFLoader básico, esta información se pierde parcialmente. La adición de pdfplumber fue la mejora de mayor impacto en la precisión de las respuestas.

**3. MMR supera a similarity_search para documentos con contenido repetitivo**  
Los cheat sheets tienen secciones temáticamente similares. MMR garantiza que los chunks recuperados cubran distintos aspectos del tema, evitando el sesgo hacia un único segmento del documento.

**4. El Grounding Check es una herramienta de diagnóstico invaluable**  
Permite identificar respuestas potencialmente alucinadas de forma automática, sin necesidad de verificación manual. Un score bajo es una señal de alerta que invita a reformular la pregunta o revisar la calidad del retrieval.

**5. Temperatura baja (0.1) reduce drásticamente las alucinaciones**  
A menor temperatura, el modelo es más conservador y más fiel al contexto. Para sistemas RAG donde la fidelidad al documento es crítica, la temperatura debe ser tan baja como sea posible sin afectar la fluidez.

**6. Los prompts graduales funcionan mejor que los binarios**  
El prompt "responde solo del contexto o di que no sabes" es demasiado rígido. El prompt v6 con tres niveles de confianza (alta/media/sin info) produce respuestas más útiles y calibradas.

---

### 🔮 Trabajo futuro

- **RAG con reranking:** Añadir un cross-encoder para reordenar chunks después de la recuperación inicial
- **Embeddings especializados en tablas:** Explorar modelos como TAPAS o Table-BERT
- **Evaluación automática de respuestas:** Usar ROUGE, BLEU o métricas de faithfulness como RAGAS
- **Retrieval híbrido:** Combinar búsqueda semántica (vectorial) con búsqueda léxica (BM25) para mejor cobertura
- **Metadatos enriquecidos:** Añadir sección del documento (e.g., "Exceptions", "String Methods") como metadato de filtrado


---

### 🔴 Detener el servidor Gradio

Ejecuta la celda siguiente para liberar el puerto y cerrar la interfaz web.


In [19]:
gr.close_all()
try:
    rag_app_v6.close()
    print("✅ Servidor Gradio cerrado correctamente.")
except Exception:
    print("✅ Servidor ya estaba cerrado.")


Closing server running on port: 7866
✅ Servidor Gradio cerrado correctamente.
